In [4]:
import kagglehub
import os
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten


In [5]:
path = kagglehub.dataset_download("samdeeplearning/deepnlp")
print("Path to dataset files:", path)

100%|██████████| 234k/234k [00:00<00:00, 54.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/samdeeplearning/deepnlp/versions/1


In [6]:
files = os.listdir(path)
print("Files in dataset:", files)

Files in dataset: ['Sheet_1.csv', 'Sheet_2.csv']


In [7]:
csv_file = [f for f in files if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))

In [8]:
df.head()

,response_id,class,response_text,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,response_1,not_flagged,I try and avoid this sort of conflict,NaN,NaN,NaN,NaN,NaN
1,response_2,flagged,Had a friend open up to me about his mental ad...,NaN,NaN,NaN,NaN,NaN
2,response_3,flagged,I saved a girl from suicide once. She was goin...,NaN,NaN,NaN,NaN,NaN
3,response_4,not_flagged,i cant think of one really...i think i may hav...,NaN,NaN,NaN,NaN,NaN
4,response_5,not_flagged,Only really one friend who doesn't fit into th...,,NaN,NaN,NaN,NaN


In [9]:
text_col = df.columns[0]
label_col = df.columns[1]

texts = df[text_col].astype(str)
labels = df[label_col].astype('category').cat.codes

In [10]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

In [11]:
max_len = 100
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42
)

In [13]:
model = Sequential([
    Embedding(input_dim=5000, output_dim=64, input_length=max_len),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 259ms/step - accuracy: 0.6406 - loss: 0.6623 - val_accuracy: 0.8750 - val_loss: 0.4560
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.6406 - loss: 0.6644 - val_accuracy: 0.8750 - val_loss: 0.4609
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6406 - loss: 0.6649 - val_accuracy: 0.8750 - val_loss: 0.5278
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.6406 - loss: 0.6550 - val_accuracy: 0.8750 - val_loss: 0.5642
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.6406 - loss: 0.6566 - val_accuracy: 0.8750 - val_loss: 0.5684


In [15]:
print("\nModel Summary:")
model.summary()

loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)


Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (32, 100, 64)          │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (32, 6400)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (32, 64)               │       409,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,189,189 (8.35 MB)

 Trainable params: 729,729 (2.78 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,459,460 (5.57 MB)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8750 - loss: 0.5684

Test Accuracy: 0.875
